# Dynamic Sign Recognition — MobileNetV3 Architecture
MediaPipe Hands → MobileNetV3 spatial features → Transformer temporal classifier

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Install required packages
# ────────────────────────────────────────────────────────────────────────────────
import subprocess
import sys

packages = [
    'torch',
    'torchvision',
    'timm',
    'mediapipe',
    'opencv-python',
    'numpy',
    'pandas',
    'matplotlib',
    'scikit-learn',
    'tqdm',
    'kagglehub'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✓ All packages installed')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Imports and CUDA check
# ────────────────────────────────────────────────────────────────────────────────
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import cv2
import timm
from timm.models import create_model
from tqdm import tqdm
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from torchvision import transforms
import warnings
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'CUDA capability: {torch.cuda.get_device_capability(0)}')
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
else:
    print('WARNING: CUDA not available, will use CPU (slow)')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Global configuration
# ────────────────────────────────────────────────────────────────────────────────

# Set working directory to script's folder (safe for multiple architecture folders)
os.chdir(os.path.dirname(os.path.abspath('mobilenetv3_notebook.ipynb')) or '.')
print(f'Working directory: {os.getcwd()}')

CONFIG = {
    # ────── IPN Dataset Paths ──────
    'IPN_ROOT': r'C:\Users\johnr\.cache\kagglehub\datasets\soumicksarker\ipn-hand-dataset\versions\7',
    'ANNOTATION_FILE': r'C:\Users\johnr\.cache\kagglehub\datasets\soumicksarker\ipn-hand-dataset\versions\7\annotation_ipnGesture\ipnall.json',
    'FRAME_DIR': r'C:\Users\johnr\.cache\kagglehub\datasets\soumicksarker\ipn-hand-dataset\versions\7\frames',
    
    # ────── Architecture: MobileNetV3 ──────
    'BACKBONE': 'mobilenetv3_large_100',
    'FEATURE_DIM': 960,  # MobileNetV3 output dimension
    'IMG_SIZE': (224, 224),  # Standard ImageNet size
    'FREEZE_LAYERS': ['features.0', 'features.1', 'features.2', 'features.3',
                      'features.4', 'features.5', 'features.6', 'features.7',
                      'features.8', 'features.9', 'features.10'],
    
    # ────── Transformer ──────
    'TRANSFORMER_HIDDEN': 256,
    'TRANSFORMER_HEADS': 4,
    'TRANSFORMER_LAYERS': 2,
    'TRANSFORMER_DROPOUT': 0.1,
    'MAX_SEQUENCE_LEN': 64,
    
    # ────── Training ──────
    'BATCH_SIZE': 16,
    'NUM_EPOCHS': 50,
    'LEARNING_RATE': 0.0001,
    'WEIGHT_DECAY': 1e-5,
    'WARMUP_EPOCHS': 5,
    'NUM_WORKERS': 4,
    'PIN_MEMORY': True,
    'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # ────── Data splits ──────
    'TRAIN_SPLIT': 0.7,
    'VAL_SPLIT': 0.15,
    'TEST_SPLIT': 0.15,
    
    # ────── Checkpoint & cache ──────
    'CHECKPOINT_DIR': './checkpoints',
    'FEATURE_CACHE_DIR': './feature_cache',
    'HAND_LANDMARKER_MODEL': './hand_landmarker.task',
    
    # ────── Class weights (inverse frequency for loss) ──────
    'CLASS_WEIGHTS': torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0, 0.3], dtype=torch.float32),  # D0X is over-represented
}

# Create directories
os.makedirs(CONFIG['CHECKPOINT_DIR'], exist_ok=True)
os.makedirs(CONFIG['FEATURE_CACHE_DIR'], exist_ok=True)

print(f'✓ Config loaded: {CONFIG["BACKBONE"]} backbone')
print(f'  Device: {CONFIG["DEVICE"]}')
print(f'  Checkpoint dir: {CONFIG["CHECKPOINT_DIR"]}')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# IPNDataset: Load IPN-hand dataset with feature caching
# ────────────────────────────────────────────────────────────────────────────────

class IPNDataset(Dataset):
    def __init__(self, annotation_file, frame_dir, feature_cache_dir, img_size, split='train'):
        self.frame_dir = frame_dir
        self.feature_cache_dir = feature_cache_dir
        self.img_size = img_size
        self.split = split
        
        with open(annotation_file, 'r') as f:
            data = json.load(f)
        
        # Class mapping (5 gestures + 1 non-gesture)
        self.class_names = ['B0A', 'G03', 'G04', 'G06', 'G07', 'D0X']
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}
        
        # Parse JSON and split by fold
        self.samples = []
        seen = set()
        
        for key, label in data.items():
            # Key format: ./frames/1CM1_4_R_#229^3
            video_id = key.split('^')[0].replace('./frames/', '')
            
            if video_id in seen:
                continue
            seen.add(video_id)
            
            # Extract fold number (e.g., 1CM1_1_R → fold 1)
            try:
                fold = int(video_id.split('_')[1])
            except:
                continue
            
            # Split by fold: folds 1-3 train, 4-5 val, rest test
            if split == 'train' and fold in [1, 2, 3]:
                self.samples.append((video_id, label))
            elif split == 'val' and fold in [4]:
                self.samples.append((video_id, label))
            elif split == 'test' and fold in [5]:
                self.samples.append((video_id, label))
        
        print(f'Loaded {len(self.samples)} {split} samples from JSON')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        video_id, label_str = self.samples[idx]
        label_idx = self.class_to_idx[label_str]
        
        video_dir = os.path.join(self.frame_dir, video_id)
        frame_files = sorted([f for f in os.listdir(video_dir) if f.endswith('.jpg')])
        
        if len(frame_files) == 0:
            return torch.zeros(CONFIG['FEATURE_DIM']), torch.tensor(label_idx)
        
        frames = []
        for fname in frame_files:
            img = cv2.imread(os.path.join(video_dir, fname))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, self.img_size)
                frames.append(img)
        
        if len(frames) == 0:
            return torch.zeros(CONFIG['FEATURE_DIM']), torch.tensor(label_idx)
        
        frames = np.stack(frames, axis=0)  # (T, H, W, 3)
        frames = torch.from_numpy(frames).float() / 255.0
        frames = frames.permute(0, 3, 1, 2)  # (T, 3, H, W)
        
        return frames, torch.tensor(label_idx)

# ── Sanity check ──
dataset_train = IPNDataset(
    CONFIG['ANNOTATION_FILE'],
    CONFIG['FRAME_DIR'],
    CONFIG['FEATURE_CACHE_DIR'],
    CONFIG['IMG_SIZE'],
    split='train'
)
print(f'✓ Train dataset: {len(dataset_train)} samples')
print(f'  Classes: {dataset_train.class_names}')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# MobileNetV3FeatureExtractor: Backbone + custom head
# ────────────────────────────────────────────────────────────────────────────────

class MobileNetV3FeatureExtractor(nn.Module):
    def __init__(self, device: str = "cpu"):
        super().__init__()
        # Load MobileNetV3 from timm with num_classes=0 to get features
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=0)
        
        # Freeze early layers for fine-tuning
        for p in self.backbone.parameters():
            p.requires_grad = False
        
        # Unfreeze later blocks
        for name, p in self.backbone.named_parameters():
            if 'features.15' in name or 'features.16' in name:
                p.requires_grad = True
        
        # Preprocessing pipeline
        self.preprocess = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(CONFIG['IMG_SIZE']),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        
        n_trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in self.backbone.parameters())
        print(f"MobileNetV3: {n_trainable:,} / {n_total:,} params trainable")
        
        self.to(device)
    
    @torch.no_grad()
    def encode(self, crop_bgr: np.ndarray) -> torch.Tensor | None:
        """Inference mode (no grad)."""
        if crop_bgr is None or crop_bgr.size == 0:
            return None
        was_training = self.backbone.training
        self.backbone.eval()
        rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        t = self.preprocess(rgb).unsqueeze(0).to(next(self.backbone.parameters()).device)
        out = self.backbone(t).squeeze(0).cpu()
        if was_training:
            self.backbone.train()
        return out
    
    def encode_train(self, crop_bgr: np.ndarray) -> torch.Tensor | None:
        """Training mode (with grad)."""
        if crop_bgr is None or crop_bgr.size == 0:
            return None
        rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        t = self.preprocess(rgb).unsqueeze(0).to(next(self.backbone.parameters()).device)
        return self.backbone(t).squeeze(0).cpu()

print('Loading MobileNetV3 weights (~20 MB)...')
spatial_extractor = MobileNetV3FeatureExtractor(device=CONFIG['DEVICE'])
print(f'✓ MobileNetV3 feature extractor ready on {CONFIG["DEVICE"]}')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# TransformerTemporalClassifier: Temporal reasoning on feature sequences
# ────────────────────────────────────────────────────────────────────────────────

class TransformerTemporalClassifier(nn.Module):
    def __init__(self, feature_dim, hidden_dim, num_heads, num_layers, num_classes, dropout=0.1, max_seq_len=64):
        super().__init__()
        self.feature_dim = feature_dim
        self.hidden_dim = hidden_dim
        
        # Project features to transformer hidden dim
        self.input_projection = nn.Linear(feature_dim, hidden_dim)
        
        # Positional encoding
        self.positional_encoding = nn.Embedding(max_seq_len, hidden_dim)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, x, mask=None):
        # x: (B, T, feature_dim)
        seq_len = x.shape[1]
        
        # Project to hidden dim
        x = self.input_projection(x)  # (B, T, hidden_dim)
        
        # Add positional encoding
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        pos_enc = self.positional_encoding(positions)  # (1, T, hidden_dim)
        x = x + pos_enc
        
        # Transformer
        x = self.transformer_encoder(x, src_key_padding_mask=mask)  # (B, T, hidden_dim)
        
        # Use CLS token (first element) for classification
        x = x[:, 0, :]  # (B, hidden_dim)
        
        # Classify
        logits = self.classifier(x)  # (B, num_classes)
        return logits

# ── Test ──
test_classifier = TransformerTemporalClassifier(
    feature_dim=CONFIG['FEATURE_DIM'],
    hidden_dim=CONFIG['TRANSFORMER_HIDDEN'],
    num_heads=CONFIG['TRANSFORMER_HEADS'],
    num_layers=CONFIG['TRANSFORMER_LAYERS'],
    num_classes=len(dataset_train.class_names),
    dropout=CONFIG['TRANSFORMER_DROPOUT'],
    max_seq_len=CONFIG['MAX_SEQUENCE_LEN']
).to(CONFIG['DEVICE'])
print(f'✓ Transformer classifier initialized')
print(f'  Hidden dim: {CONFIG["TRANSFORMER_HIDDEN"]}')
print(f'  Heads: {CONFIG["TRANSFORMER_HEADS"]}')
print(f'  Layers: {CONFIG["TRANSFORMER_LAYERS"]}')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# DynamicSignModel: End-to-end wrapper
# ────────────────────────────────────────────────────────────────────────────────

class DynamicSignModel(nn.Module):
    def __init__(self, feature_extractor, classifier):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.classifier = classifier
    
    def forward(self, x):
        # x: (B, T, 3, H, W)
        B, T, C, H, W = x.shape
        
        # Reshape to (B*T, 3, H, W) for batch processing
        x = x.reshape(B * T, C, H, W)
        
        # Extract features
        features = self.feature_extractor(x)  # (B*T, feature_dim)
        
        # Reshape back to (B, T, feature_dim)
        features = features.reshape(B, T, -1)
        
        # Classify
        logits = self.classifier(features)  # (B, num_classes)
        return logits

# ── Instantiate full model ──
feature_extractor = MobileNetV3FeatureExtractor(
    feature_dim=CONFIG['FEATURE_DIM'],
    freeze_layers=CONFIG['FREEZE_LAYERS']
)

classifier = TransformerTemporalClassifier(
    feature_dim=CONFIG['FEATURE_DIM'],
    hidden_dim=CONFIG['TRANSFORMER_HIDDEN'],
    num_heads=CONFIG['TRANSFORMER_HEADS'],
    num_layers=CONFIG['TRANSFORMER_LAYERS'],
    num_classes=len(dataset_train.class_names),
    dropout=CONFIG['TRANSFORMER_DROPOUT'],
    max_seq_len=CONFIG['MAX_SEQUENCE_LEN']
)

model = DynamicSignModel(feature_extractor, classifier).to(CONFIG['DEVICE'])
print(f'✓ Full model instantiated and moved to {CONFIG["DEVICE"]}')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Training utilities: Metric tracking and epoch functions
# ────────────────────────────────────────────────────────────────────────────────

class AverageMeter:
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def train_epoch(model, train_loader, criterion, optimizer, device, scaler=None):
    model.train()
    losses = AverageMeter()
    accuracies = AverageMeter()
    
    with tqdm(total=len(train_loader), desc='Train', leave=False) as pbar:
        for frames, labels in train_loader:
            frames = frames.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            if scaler:
                with torch.autocast(device_type='cuda'):
                    logits = model(frames)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(frames)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()
            
            # Metrics
            preds = torch.argmax(logits, dim=1)
            acc = (preds == labels).float().mean().item()
            
            losses.update(loss.item(), frames.size(0))
            accuracies.update(acc, frames.size(0))
            
            pbar.update(1)
            pbar.set_postfix({'loss': f'{losses.avg:.4f}', 'acc': f'{accuracies.avg:.4f}'})
    
    return losses.avg, accuracies.avg

def evaluate(model, val_loader, criterion, device):
    model.eval()
    losses = AverageMeter()
    accuracies = AverageMeter()
    
    with torch.no_grad():
        with tqdm(total=len(val_loader), desc='Val', leave=False) as pbar:
            for frames, labels in val_loader:
                frames = frames.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                logits = model(frames)
                loss = criterion(logits, labels)
                
                preds = torch.argmax(logits, dim=1)
                acc = (preds == labels).float().mean().item()
                
                losses.update(loss.item(), frames.size(0))
                accuracies.update(acc, frames.size(0))
                
                pbar.update(1)
                pbar.set_postfix({'loss': f'{losses.avg:.4f}', 'acc': f'{accuracies.avg:.4f}'})
    
    return losses.avg, accuracies.avg

print('✓ Training utilities defined')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Full training loop with oversampling (NEW: replaces WeightedRandomSampler)
# ────────────────────────────────────────────────────────────────────────────────

def train(model, train_dataset, val_dataset, config, start_epoch=0):
    device = config['DEVICE']
    num_epochs = config['NUM_EPOCHS']
    batch_size = config['BATCH_SIZE']
    num_workers = config['NUM_WORKERS']
    
    # ────── OVERSAMPLING: Balance minority classes by duplication ──────
    from collections import Counter
    class_counts = Counter([train_dataset.samples[i][1] for i in range(len(train_dataset))])
    max_count = max(class_counts.values())
    
    indices = []
    for idx in range(len(train_dataset)):
        label = train_dataset.samples[idx][1]
        times_to_repeat = max_count // class_counts[label]
        indices.extend([idx] * times_to_repeat)
    
    balanced_train_dataset = Subset(train_dataset, indices)
    
    print(f'Oversampling: {len(train_dataset)} → {len(balanced_train_dataset)} samples')
    print(f'  Class counts (original): {dict(class_counts)}')
    
    # DataLoaders
    train_loader = DataLoader(
        balanced_train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=config['PIN_MEMORY']
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=config['PIN_MEMORY']
    )
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(weight=config['CLASS_WEIGHTS'].to(device))
    optimizer = optim.AdamW(model.parameters(), lr=config['LEARNING_RATE'], weight_decay=config['WEIGHT_DECAY'])
    
    # Learning rate schedule with warmup
    def get_lr(epoch):
        if epoch < config['WARMUP_EPOCHS']:
            return config['LEARNING_RATE'] * (epoch + 1) / config['WARMUP_EPOCHS']
        else:
            return config['LEARNING_RATE'] * 0.5 ** ((epoch - config['WARMUP_EPOCHS']) // 10)
    
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, get_lr)
    scaler = torch.cuda.amp.GradScaler() if device == 'cuda' else None
    
    best_val_acc = 0.0
    checkpoint_path = os.path.join(config['CHECKPOINT_DIR'], 'best_model.pt')
    
    # ────── NEW: Track training history ──────
    training_history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    for epoch in range(start_epoch, num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
        
        # Validate
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        # Record history
        training_history['train_loss'].append(train_loss)
        training_history['train_acc'].append(train_acc)
        training_history['val_loss'].append(val_loss)
        training_history['val_acc'].append(val_acc)
        
        # LR schedule
        scheduler.step()
        
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
        print(f'LR: {optimizer.param_groups[0]["lr"]:.6f}')
        
        # Save best checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_acc': best_val_acc,
                'backbone': config['BACKBONE']
            }, checkpoint_path)
            print(f'✓ Checkpoint saved (acc: {best_val_acc:.4f})')
    
    print(f'\n✓ Training complete. Best val acc: {best_val_acc:.4f}')
    
    # ────── NEW: Return training history ──────
    return training_history

print('✓ Training function defined')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Load all datasets and start training
# ────────────────────────────────────────────────────────────────────────────────

print('Loading datasets...')
dataset_train = IPNDataset(CONFIG['ANNOTATION_FILE'], CONFIG['FRAME_DIR'], CONFIG['FEATURE_CACHE_DIR'], CONFIG['IMG_SIZE'], split='train')
dataset_val = IPNDataset(CONFIG['ANNOTATION_FILE'], CONFIG['FRAME_DIR'], CONFIG['FEATURE_CACHE_DIR'], CONFIG['IMG_SIZE'], split='val')
dataset_test = IPNDataset(CONFIG['ANNOTATION_FILE'], CONFIG['FRAME_DIR'], CONFIG['FEATURE_CACHE_DIR'], CONFIG['IMG_SIZE'], split='test')

print(f'Train: {len(dataset_train)} | Val: {len(dataset_val)} | Test: {len(dataset_test)}')

# Start training
training_history = train(model, dataset_train, dataset_val, CONFIG)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# MediaPipe setup: Hand detection + landmark extraction + palm check
# ────────────────────────────────────────────────────────────────────────────────

def download_hand_landmarker_model():
    """Download hand_landmarker.task if not present."""
    model_path = CONFIG['HAND_LANDMARKER_MODEL']
    if not os.path.exists(model_path):
        print('Downloading hand_landmarker.task...')
        import urllib.request
        url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
        urllib.request.urlretrieve(url, model_path)
        print(f'✓ Saved to {model_path}')

download_hand_landmarker_model()

def build_hands_tracker() -> mp_vision.HandLandmarker:
    """Return a fresh HandLandmarker in VIDEO mode."""
    opts = mp_vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=CONFIG['HAND_LANDMARKER_MODEL']),
        num_hands=1,
        min_hand_detection_confidence=0.60,
        min_hand_presence_confidence=0.50,
        min_tracking_confidence=0.50,
        running_mode=mp_vision.RunningMode.VIDEO,
    )
    return mp_vision.HandLandmarker.create_from_options(opts)

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(17,18),(18,19),(19,20),
    (0,17),
]

def draw_hand(frame_bgr: np.ndarray, hand_landmarks_list) -> None:
    """Draw hand skeleton on frame."""
    h, w = frame_bgr.shape[:2]
    for hand_lms in hand_landmarks_list:
        pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_lms]
        for a, b in HAND_CONNECTIONS:
            cv2.line(frame_bgr, pts[a], pts[b], (255, 80, 0), 2)
        for pt in pts:
            cv2.circle(frame_bgr, pt, 3, (0, 255, 120), -1)

def get_hand_bbox(frame: np.ndarray, hand_landmarks, pad: int = 30):
    """Return (crop, bbox) for the detected hand."""
    h, w = frame.shape[:2]
    xs = [lm.x * w for lm in hand_landmarks]
    ys = [lm.y * h for lm in hand_landmarks]
    cx = (min(xs) + max(xs)) / 2
    cy = (min(ys) + max(ys)) / 2
    half = max(max(xs) - min(xs), max(ys) - min(ys)) / 2 + pad
    x0, x1 = int(max(0, cx - half)), int(min(w, cx + half))
    y0, y1 = int(max(0, cy - half)), int(min(h, cy + half))
    return frame[y0:y1, x0:x1], (x0, y0, x1, y1)

FINGER_TIPS = [8, 12, 16, 20]
FINGER_PIPS = [6, 10, 14, 18]

def is_open_palm(hand_landmarks, handedness: str = "Right") -> bool:
    """Check if hand is in open-palm pose."""
    lm = hand_landmarks
    fingers_up = sum(lm[tip].y < lm[pip].y for tip, pip in zip(FINGER_TIPS, FINGER_PIPS))
    thumb_abducted = (lm[4].x > lm[2].x) if handedness == "Right" else (lm[4].x < lm[2].x)
    return fingers_up >= 3 and thumb_abducted

print('✓ MediaPipe utilities defined')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Classify a collected clip
# ────────────────────────────────────────────────────────────────────────────────

def classify_clip(model, frames, device, img_size):
    """Classify a clip and return gesture label + confidence."""
    model.eval()
    
    # Preprocess frames
    processed_frames = []
    for frame in frames:
        if frame.shape != img_size:
            frame = cv2.resize(frame, img_size)
        frame = frame.astype(np.float32) / 255.0
        processed_frames.append(frame)
    
    frames_tensor = torch.from_numpy(np.stack(processed_frames)).float()
    frames_tensor = frames_tensor.permute(0, 3, 1, 2).unsqueeze(0)  # (1, T, 3, H, W)
    frames_tensor = frames_tensor.to(device)
    
    with torch.no_grad():
        logits = model(frames_tensor)
    
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred_idx = np.argmax(probs)
    confidence = probs[pred_idx]
    
    return pred_idx, confidence, probs

print('✓ Clip classification function defined')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Load best checkpoint
# ────────────────────────────────────────────────────────────────────────────────

checkpoint_path = os.path.join(CONFIG['CHECKPOINT_DIR'], 'best_model.pt')
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=CONFIG['DEVICE'])
    model.load_state_dict(checkpoint['model_state'])
    print(f'✓ Loaded checkpoint (epoch {checkpoint["epoch"]}, acc {checkpoint["best_acc"]:.4f})')
else:
    print('⚠ No checkpoint found, using untrained model')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Live webcam demo
# ────────────────────────────────────────────────────────────────────────────────

hands = build_hands_tracker()
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

frame_count = 0
prediction_text = ''
confidence_text = ''

print('\n🎥 Webcam running (press Q to quit)...')
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    
    # Detect hands
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    result = hands.detect_for_video(mp_image, frame_count * 33)  # ~30fps
    
    frame_count += 1
    
    # Draw landmarks if detected
    if result.hand_landmarks:
        hand_lm = result.hand_landmarks[0]
        handedness = result.handedness[0][0].category_name if result.handedness else "Right"
        draw_hand(frame, result.hand_landmarks)
        
        crop, bbox = get_hand_bbox(frame, hand_lm)
        triggered = is_open_palm(hand_lm, handedness)
        
        if triggered:
            # Encode crop
            feat = spatial_extractor.encode(crop)
            if feat is not None:
                prediction_text = f'Gesture detected (inference ready)'
    
    # Draw status
    status = 'PALM DETECTED' if triggered else 'WAITING...'
    cv2.putText(frame, status, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
    
    if prediction_text:
        cv2.putText(frame, prediction_text, (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
    
    cv2.imshow('Dynamic Sign Recognition', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print('✓ Webcam closed')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# Evaluate on test set + plot results (NEW: from training_history)
# ────────────────────────────────────────────────────────────────────────────────

# ────── Plot training curves ──────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(training_history['train_loss'], label='Train', marker='o')
axes[0].plot(training_history['val_loss'], label='Val', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(training_history['train_acc'], label='Train', marker='o')
axes[1].plot(training_history['val_acc'], label='Val', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['CHECKPOINT_DIR'], 'training_curves.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'✓ Curves saved to {CONFIG["CHECKPOINT_DIR"]}/training_curves.png')

# ────── Evaluate on test set ──────
test_loader = DataLoader(dataset_test, batch_size=CONFIG['BATCH_SIZE'], num_workers=CONFIG['NUM_WORKERS'], pin_memory=True)
criterion = nn.CrossEntropyLoss(weight=CONFIG['CLASS_WEIGHTS'].to(CONFIG['DEVICE']))
test_loss, test_acc = evaluate(model, test_loader, criterion, CONFIG['DEVICE'])

print(f'\n✓ Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}')

# ────── Confusion matrix ──────
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for frames, labels in test_loader:
        frames = frames.to(CONFIG['DEVICE'], non_blocking=True)
        labels = labels.to(CONFIG['DEVICE'], non_blocking=True)
        logits = model(frames)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print(f'\nConfusion Matrix:\n{cm}')

# Classification report (NEW: formatted table output)
print(f'\nDetailed Classification Report:')
report_dict = classification_report(all_labels, all_preds, target_names=dataset_train.class_names, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
print(report_df.to_string())

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
ax.set_xticks(range(len(dataset_train.class_names)))
ax.set_yticks(range(len(dataset_train.class_names)))
ax.set_xticklabels(dataset_train.class_names)
ax.set_yticklabels(dataset_train.class_names)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['CHECKPOINT_DIR'], 'confusion_matrix.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'✓ Confusion matrix saved')